In [1]:
import random
import ollama
from pydantic import BaseModel
from typing import Literal
from api.listening.listening_service import ListeningService
from common.ko_util import korean_to_english_pronunciation
from db.model.scenario import StageType, QuestLevel, ReadingQuest, ListeningQuest

class WordData(BaseModel):
    kor: str
    eng: str
    pronunciation: str
class TargetItem(BaseModel):
    name:str
    code:str
class TargetData(BaseModel):
    word1:TargetItem
    word2:TargetItem
class QuestBase(BaseModel):
    index:int
    difficulty:QuestLevel
    room_id:int
    
## Reading, Listening Scenario Quest Info    
class QuestReadInfo(QuestBase):
    target_data: list[TargetData]
    correct_answer_index: int
    word_data1: WordData
    word_data2: WordData
    full_data: WordData

class QuestListenInfo(QuestReadInfo):
    voice_data: str
    
## Writing Scenario Quest Info
class WriteData(BaseModel):
    word_data: WordData
    answer: str
    answer_kor: str

class QuestWriteInfo(QuestBase):
    question: list[WriteData]

## Speaking Scenario Quest Info
class SpeakData(WordData):
    voice_data: str | None = None

class QuestSpeakInfo(QuestBase):
    audio: list[SpeakData]

2025-12-19 11:45:35.109 | INFO     | db.session:<module>:20 - JWT setup
2025-12-19 11:45:35.745 | INFO     | db.session:<module>:29 - JWT Utility Function


In [2]:
tts = ListeningService()

### quest type, level에 해당하는 quest 정보 가져오기
def quest_words(quests:list[ReadingQuest | ListeningQuest],_type:str,level:QuestLevel):
    words = []
    for word in [q.quest_words for q in quests if q.quest_type == _type and q.quest_level == level]:
        words.extend(word)
    return words

### 읽기, 듣기 시나리오 생성용
def ko_to_en(ko:str):
    system_prompt = """
        당신은 **영어** 번역 전문가 입니다.
        한글 문장을 영문으로 번역하여 **번역된 영문**만 알려주세요.
        다른 설명이나 인사는 **절대로** 포함하지 마세요.
    """
    system_prompt = """당신은 한국어 문장을 **영어**로 번역하는 전문 영어 번역가입니다. 
            번역할 한국어 문장을 입력받으면, 해당 문장의 **영어 번역 결과**만 출력하고
            다른 설명이나 추가 문장은 절대 포함하지 마세요."""
    user_prompt = "한글 문장을 영문으로 번역해줘 : {}"

    ko_sample1 = "하늘이 매우 파랗습니다."
    en_answer1 = "The sky is very blue."
    ko_sample2 = "나는 사과를 먹었다."
    en_answer2 = "He ate a apple."
    response = ollama.chat(
        model="hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M",
        messages=[
            {'role': 'system', 'content': system_prompt},
            ## 예시
            {'role': 'user', 'content': user_prompt.format(ko_sample1)},
            {'role': 'assistant', 'content': en_answer1},
            {'role': 'user', 'content': user_prompt.format(ko_sample2)},
            {'role': 'assistant', 'content': en_answer2},
            ## 사용자 정보
            {'role': 'user', 'content': user_prompt.format(ko)}
        ]
    )
    return response['message']['content']

quest_template = {
    StageType.READING:{
        'word_data1':"{} 스티커를 찾아라",
        'word_data2':"{} 캐리어를 찾아라",
        'full_data':"{} 스티커가 붙은 {} 캐리어를 찾아라"
    },
    StageType.LISTENING:{
        'word_data1':"{}에서 찾아라",
        'word_data2':"제일 맛있는 {} 집을 찾아라",
        'full_data':"{}에서 제일 맛있는 {} 집을 찾아라"
    }
}

Device set to use cuda:0


In [ ]:
def quest_items(quests:list[BaseModel],_type:str,level:QuestLevel):
    ## 해당 레벨의 퀘스트 갯수만큼 샘플링    
    items = []
    for item_zip in [zip(q.quest_codes,q.quest_words)
                 for q in quests if q.quest_type == _type and q.quest_level == level]:
        for item in item_zip:
            items.append(TargetItem(code=item[0],name=item[1]))
    return items

def quest_sampling(stage_type:Literal[StageType.READING, StageType.LISTENING],
        quests:list[BaseModel],level:QuestLevel,count:int = 10):
    """
        레벨에 따라 샘플링 갯수 조정
        Level : EASY - 100%, NORMAL - EASY 50%, NORMAL 50%, HARD - EASY 20%, NORMAL 30%, HARD 50%
    """
    sampling_count = {
        QuestLevel.EASY:0,
        QuestLevel.NORMAL:0,
        QuestLevel.HARD:0
    }
    if level == QuestLevel.EASY:
        sampling_count[QuestLevel.EASY] = count
    elif level == QuestLevel.NORMAL:
        sampling_count[QuestLevel.NORMAL] = int(count * 0.5)
        sampling_count[QuestLevel.EASY] = count - sampling_count[QuestLevel.NORMAL]
    else:
        sampling_count[QuestLevel.HARD] = int(count * 0.5)
        sampling_count[QuestLevel.NORMAL] = int((count - sampling_count[QuestLevel.HARD]) * 0.6)
        sampling_count[QuestLevel.EASY] = count - sampling_count[QuestLevel.HARD] - sampling_count[QuestLevel.NORMAL]
    # print(sampling_count)
    word1_type = 'symbol' if stage_type == StageType.READING else 'region'
    word2_type = 'color' if stage_type == StageType.READING else 'food'
    sampling_list = []
    for _sampling_level in sampling_count:
        if sampling_count[_sampling_level] <= 0:
            continue
        word1 = quest_items(quests,word1_type,_sampling_level)
        word2 = quest_items(quests,word2_type,_sampling_level)
        sampling_list.extend(random.sample([(w1, w2) for w1 in word1 for w2 in word2],sampling_count[_sampling_level]))
    return random.sample(sampling_list,count)

In [30]:
from masterdata.reading_data import reading_data
stage_type = StageType.READING # StageType.LISTENING]
quests = [ReadingQuest(**d) for d in reading_data]

In [33]:
quest_sampling(stage_type,quests,QuestLevel.HARD)

{<QuestLevel.EASY: 1>: 2, <QuestLevel.NORMAL: 2>: 3, <QuestLevel.HARD: 3>: 5}


[(TargetItem(name='코끼리', code='18'), TargetItem(name='황토색', code='19')),
 (TargetItem(name='곰', code='13'), TargetItem(name='분홍', code='9')),
 (TargetItem(name='사슴', code='11'), TargetItem(name='금색', code='13')),
 (TargetItem(name='치타', code='19'), TargetItem(name='군청색', code='21')),
 (TargetItem(name='고양이', code='2'), TargetItem(name='파랑', code='2')),
 (TargetItem(name='곰', code='13'), TargetItem(name='금색', code='13')),
 (TargetItem(name='물개', code='20'), TargetItem(name='베이지', code='24')),
 (TargetItem(name='닭', code='6'), TargetItem(name='빨강', code='1')),
 (TargetItem(name='하마', code='21'), TargetItem(name='베이지', code='24')),
 (TargetItem(name='물개', code='20'), TargetItem(name='연두색', code='22'))]

In [34]:
def gen_read_or_listen_quest(stage_type: Literal[StageType.READING, StageType.LISTENING],
        quests:list[BaseModel],level:QuestLevel,quest_count:int = 10):
    """
        quests : read quest list
        level  : quest level
        quest_count : 필요 갯수
        읽기 시나리오 생성
        Level : EASY - 100%, NORMAL - EASY 50%, NORMAL 50%, HARD - EASY 20%, NORMAL 30%, HARD 50%
    """
    # word1_type = 'symbol' if stage_type == StageType.READING else 'region'
    # word2_type = 'color' if stage_type == StageType.READING else 'food'
    # word1 = quest_items(quests,word1_type,level)
    # word2 = quest_items(quests,word2_type,level)
    # quest_data = random.sample([(w1, w2) for w1 in word1 for w2 in word2],quest_count)
    quest_data = quest_sampling(stage_type,quests,level)
    correct_index = random.randint(0,quest_count-1)
    target_data = [TargetData(word1=q_data[0],word2=q_data[1]) for q_data in quest_data]
    word_data1 = quest_template[stage_type]['word_data1'].format(quest_data[correct_index][0].name)
    word_data2 = quest_template[stage_type]['word_data2'].format(quest_data[correct_index][1].name)
    full_data = quest_template[stage_type]['full_data'].format(quest_data[correct_index][0].name,quest_data[correct_index][1].name)
    if stage_type == StageType.READING:
        return QuestReadInfo(
            index=1,
            difficulty=level,
            room_id=0,
            target_data=target_data,
            correct_answer_index=correct_index,
            word_data1=WordData(
                kor = word_data1,
                eng = ko_to_en(word_data1),
                pronunciation=korean_to_english_pronunciation(word_data1)
            ),
            word_data2=WordData(
                kor = word_data2,
                eng = ko_to_en(word_data2),
                pronunciation=korean_to_english_pronunciation(word_data2)
            ),
            full_data=WordData(
                kor = full_data,
                eng = ko_to_en(full_data),
                pronunciation=korean_to_english_pronunciation(full_data)
            )
        )
    else: #if stage_type == StageType.LISTENING:
        return QuestListenInfo(
            index=1,
            difficulty=level,
            room_id=0,
            target_data=target_data,
            correct_answer_index=correct_index,
            word_data1=WordData(
                kor = word_data1,
                eng = ko_to_en(word_data1),
                pronunciation=korean_to_english_pronunciation(word_data1)
            ),
            word_data2=WordData(
                kor = word_data2,
                eng = ko_to_en(word_data2),
                pronunciation=korean_to_english_pronunciation(word_data2)
            ),
            full_data=WordData(
                kor = full_data,
                eng = ko_to_en(full_data),
                pronunciation=korean_to_english_pronunciation(full_data)
            ),
            voice_data = tts.make_audio_base64_from_text(full_data).audio_base64
        )

In [37]:
sample_list = gen_read_or_listen_quest(stage_type,quests,QuestLevel.HARD)
for s in sample_list.target_data:
    print(s)

{<QuestLevel.EASY: 1>: 2, <QuestLevel.NORMAL: 2>: 3, <QuestLevel.HARD: 3>: 5}
word1=TargetItem(name='낙타', code='22') word2=TargetItem(name='군청색', code='21')
word1=TargetItem(name='물개', code='20') word2=TargetItem(name='와인색', code='23')
word1=TargetItem(name='물개', code='20') word2=TargetItem(name='군청색', code='21')
word1=TargetItem(name='치타', code='19') word2=TargetItem(name='자주색', code='17')
word1=TargetItem(name='소', code='5') word2=TargetItem(name='주황', code='8')
word1=TargetItem(name='원숭이', code='15') word2=TargetItem(name='금색', code='13')
word1=TargetItem(name='말', code='9') word2=TargetItem(name='갈색', code='10')
word1=TargetItem(name='사슴', code='11') word2=TargetItem(name='은색', code='14')
word1=TargetItem(name='기린', code='17') word2=TargetItem(name='와인색', code='23')
word1=TargetItem(name='닭', code='6') word2=TargetItem(name='빨강', code='1')
